# DEMETER LSTM-AE — Evaluation of Pretrained Models

No training. For each rolling window, the saved model `best_model_Hp_{m}_w{i}.pth` is loaded and evaluated
exactly as in the training notebook (same data split, same scaler fitted on normal training pairs,
threshold at the `pc`-th percentile of the training errors, same seismic association).

The last section compares the results with a previously saved result file, to check that the
cleaned functions reproduce the original outcomes.

## Setup (PC or Google Colab)

- **PC:** set `LOCAL_PROJECT` to the repository folder.
- **Colab:** set `DRIVE_PROJECT` to the repository folder in your Google Drive. If it does not exist yet,
  it is cloned there from GitHub (code + data). An uploaded copy of the folder works too.

In [ ]:
import sys
import subprocess
from pathlib import Path

LOCAL_PROJECT = Path(r"D:\GIT\LSTM-AE-For-Demeter-Seismic-Study")
DRIVE_PROJECT = Path("/content/drive/MyDrive/LSTM-AE-For-Demeter-Seismic-Study")
REPO_URL = "https://github.com/Meghababu99/LSTM-AE-For-Demeter-Seismic-Study.git"
BRANCH = "main"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = DRIVE_PROJECT
    if not PROJECT.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(PROJECT)], check=True)
else:
    PROJECT = LOCAL_PROJECT

CODE_DIR = PROJECT / "Clean-Code"
sys.path.insert(0, str(CODE_DIR))
print("Project:", PROJECT)

In [ ]:
import random

import numpy as np
import pandas as pd
import torch

from lstm import (
    LSTMAutoencoder,
    SeismicCriteria,
    generate_windows,
    load_eq_catalog,
    prepare_window,
    train_window,
    evaluate_window,
)

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

## Settings

In [ ]:
SEED = 201894
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Data
DATA_DIR = PROJECT / "Data"
BG_WINDOW_DIR = DATA_DIR / "Bg_window_data"
LABEL_DIR = DATA_DIR / "Label_data"
EQ_PATH = DATA_DIR / "EQ.csv"

# Rolling windows
train_months, val_months, stride = 12, 3, 3
WINDOW_START, WINDOW_END = "2005-01-01 00:00:00", "2010-01-02 00:00:00"
min_data_points = 17

# Anomaly threshold (percentile) and seismic criterion
pc = 98   # percentile
tw = 48   # time window (hours)
sw = 22   # spatial width (degrees)

# Model
input_size, hidden_size, num_layers, latent_dim = 11, 8, 2, 2

m = f"tw{tw}-30dBG_A"  # tag of the saved models: best_model_Hp_{m}_w{i}.pth

MODEL_DIR = PROJECT / "Models-Trained" / f"SW{sw}_TW{tw}"
REFERENCE_CSV = PROJECT / "Result Files" / f"SW{sw}_TW{tw}" / f"Retrain_result_Hp_B-{m}.csv"

OUTPUT_DIR_RESULTS = CODE_DIR / "outputs_pretrained" / m
OUTPUT_DIR_RESULTS.mkdir(parents=True, exist_ok=True)
results_csv = OUTPUT_DIR_RESULTS / f"result_Hp_{m}.csv"
print("Models:", MODEL_DIR)
print("Outputs:", OUTPUT_DIR_RESULTS)

## Earthquake catalogue and rolling windows

In [ ]:
eq = load_eq_catalog(EQ_PATH)
seismic_criteria = SeismicCriteria(spatial_width=sw, time_window_hours=tw)
windows_train = generate_windows(WINDOW_START, WINDOW_END, train_months, val_months, stride)

print("Earthquake catalogue shape:", eq.shape)
print("Number of rolling windows:", len(windows_train))
for i, w in enumerate(windows_train):
    print(f"Window {i}: Train {w['train_start']} -> {w['train_end']} | Val {w['val_start']} -> {w['val_end']}")

## Evaluation loop (pretrained models)

In [ ]:
all_results = []

for i, w in enumerate(windows_train):
    tag = f"Hp_{m}_w{i}"
    print(f"\n{'=' * 90}\nWindow {i}: Train {w['train_start']} -> {w['train_end']} | "
          f"Val {w['val_start']} -> {w['val_end']}\n{'=' * 90}")

    splits, datasets, scaled = prepare_window(
        BG_WINDOW_DIR / f"Background_data-window_{i}.pkl",
        {s: LABEL_DIR / f"summary_df_{s.lower()}_30D-{sw}SW-tw{tw}_w{i}.csv" for s in ("Train", "Val")},
        w,
        min_data_points,
    )

    model = LSTMAutoencoder(input_size, hidden_size, num_layers, latent_dim)
    model.load_state_dict(torch.load(MODEL_DIR / f"best_model_{tag}.pth", map_location="cpu"))
    print(f"[Loaded] best_model_{tag}.pth")

    all_results += evaluate_window(i, w, model, splits, datasets, scaled, eq, seismic_criteria, pc, m, OUTPUT_DIR_RESULTS)
    pd.DataFrame(all_results).to_csv(results_csv, index=False)  # saved after every window

## Comparison with the saved result file

In [ ]:
df_results = pd.DataFrame(all_results)
reference = pd.read_csv(REFERENCE_CSV)
assert len(reference) == len(df_results), f"Row count differs: reference {len(reference)}, new {len(df_results)}"

new = df_results.assign(seismic_indices=df_results["seismic_indices"].astype(str))
checks = pd.DataFrame({
    "window": new["window"],
    "split": new["split"],
    "split_ok": new["split"] == reference["split"],
    "anomalies_ok": new["anomalies"] == reference["anomalies"],
    "agg_value_ok": np.isclose(new["agg_value"], reference["agg_value"]),
    "agg_error_ok": np.isclose(new["agg_error"], reference["agg_error"]),
    "total_eq_ok": new["agg_total_eq"] == reference["agg_total_eq"],
    "seismic_indices_ok": new["seismic_indices"] == reference["seismic_indices"],
})
checks["all_ok"] = checks.filter(like="_ok").all(axis=1)
checks.to_csv(OUTPUT_DIR_RESULTS / f"comparison_with_reference_{m}.csv", index=False)

print(f"Rows identical to the reference: {checks['all_ok'].sum()} / {len(checks)}")
display(checks[~checks["all_ok"]] if not checks["all_ok"].all() else df_results)